# Analysis of MS proteomics Ependymoma dataset
> SysBio MS Proteomics course 2026
>
> By Ian Dirk Fichtner, Dennis Friedel and Rushda Patel
> 
> Neuropathology Department Heidelberg

## Tutorial overview

This tutorial walks through a complete computational mass-spectrometry (MS) proteomics analysis on a ependymoma cohort. We start from the [DIA-NN](https://github.com/vdemichev/DiaNN) search-engine output (precursor-level `report.parquet`) and end with biological interpretation via gene-set enrichment.

**Core pipeline**

1. **Read in DIA-NN output** and assemble a sample × protein matrix.
2. **Quality control** — drop contaminants, low-coverage samples, and sparsely measured proteins.
3. **Preprocessing** — log2 transform, median-normalize, impute missing values.
4. **Exploratory analysis** — sample correlation, PCA, UMAP.
5. **Differential protein abundance** — t-test between tumor classes.
6. **Pathway-level interpretation** — gene-set enrichment with `decoupler` against MSigDB.


**📝 Your-turn prompts**

Throughout the notebook you will see blockquotes marked with a 📝 emoji and the label *Your turn*. Each one sits directly below a plot or output and asks you to **interpret what you see** — pause, look at the figure, write down a short answer before scrolling on. Discussing with a neighbour is encouraged.

## Setup

> 📂 *Before you start*: this notebook expects you to load input files. It is recommended to save place them in a `data/` directory next at the project root. Place the following files there before running the cells below:
>
> - `data/report.parquet` — DIA-NN precursor-level search output.
> - `data/prot_anno.csv` — sample annotation (`sample_id` + `group` columns).
>
> The first code cell `cd`s up to the project root, so all paths below are written relative to it (e.g. `data/report.parquet`).

### A note on `proteopy` and `AnnData`

[`proteopy`](https://proteopy.readthedocs.io/) is a Python toolbox for MS proteomics analysis based on the [`AnnData`](https://anndata.readthedocs.io/) class. It exposes the analysis stages as submodules:

- `pr.read` — generic long-format tables.
- `pr.pp` — preprocessing (contaminant removal, filtering, normalization, imputation).
- `pr.pl` — plots.
- `pr.tl` — analytical tools (e.g. differential abundance).
- `pr.get` — accessors that pull results back out as tidy data frames.
- `pr.ann` — sample/feature annotation helpers.
- `pr.download` — fetch reference resources (e.g. contaminant FASTAs).

All data live inside an [`AnnData`](https://anndata.readthedocs.io/) object. AnnData has a small set of well-defined slots:

- `.X` — the main matrix; here **samples × proteins**.
- `.obs` — a data frame of per-sample (observation) metadata.
- `.var` — a data frame of per-protein (variable) metadata.
- `.layers` — alternative matrices of the same shape as `.X` (e.g. raw vs. log-transformed intensities).
- `.varm` / `.obsm` — multi-dimensional annotations per variable / observation. We will store differential-abundance results in `.varm`.
- `.uns` — unstructured annotations.

We will print the AnnData object right after reading in the data so you can see this structure for yourself.


> 🔥 *Reference*: every `proteopy` function used below is documented in the [API reference](https://proteopy.readthedocs.io/en/latest/api/index.html). Keep it open in a tab — when something is unclear, you can find useful information here.

In [ ]:
import os
import sys
import random
from pathlib import Path

import anndata as ad
import proteopy as pr
import scanpy as sc
import decoupler as dc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns

random.seed(42)

cwd = Path('.').resolve()
root = cwd.parent
os.chdir(root)

sys.path.append(str(root / 'code'))


## Read in and format the data

### What does DIA-NN give us?

[DIA-NN](https://github.com/vdemichev/DiaNN) is a search engine for **data-independent acquisition (DIA)** mass spectrometry. For every MS run it reports — in long format — every precursor it identified, together with quantitative values, several false-discovery-rate (FDR) statistics and protein-level annotations.

Relevant columns we load:

- `Run` — the MS run / sample.
- `Protein.Ids`, `Protein.Group`, `Genes`, `Protein.Names` — protein identifiers. A **protein group** is a set of proteins that share the same peptide evidence and therefore cannot be told apart by MS alone.
- `PG.MaxLFQ` — the label-free quantification value from the MaxLFQ algorithm; this is the per-run intensity we will use.
- `Q.Value`, `Protein.Q.Value`, `Global.Q.Value` — FDR statistics at different levels (precursor in run, protein in run, protein across experiment). We will filter precursors based on these statistics.

We then collapse the long table to a sample × protein matrix with `pr.read.long` which will serve as input for the `proteopy` reader function.

In [ ]:
PARQUET_PATH = "data/report.parquet"
METADATA_PATH = "data/anno.csv"

In [ ]:
df_raw = pd.read_parquet(
    PARQUET_PATH,
    columns=["Run", "Protein.Ids", "Protein.Group", "Genes", "Protein.Names",
             "PG.MaxLFQ", "Q.Value", "Protein.Q.Value", "Global.Q.Value"],
)

df_filt = (
    df_raw[
        (df_raw["Q.Value"] <= 0.01) &
        (df_raw["Protein.Q.Value"] <= 0.01) &
        (df_raw["Global.Q.Value"] <= 0.01) &
        (df_raw["PG.MaxLFQ"].notna()) &
        (df_raw["PG.MaxLFQ"] > 0)
    ]
    .drop_duplicates(subset=["Run", "Protein.Ids"])
)

print(f"Rows after filtering: {len(df_filt)}")

df_long = df_filt[["Run", "Protein.Ids", "PG.MaxLFQ"]].copy()
df_long.columns = ["sample_id", "protein_id", "intensity"]

var_annot = (
    df_filt.drop_duplicates("Protein.Ids")
    [["Protein.Ids", "Protein.Group", "Genes", "Protein.Names"]]
    .rename(columns={"Protein.Ids": "protein_id"})
)

adata = pr.read.long(
    df_long,
    level="protein",
    var_annotation=var_annot,
    fill_na=0,
    verbose=True,
)

adata

Notice the printout above: 54 samples × 10,796 proteins. `.obs` carries the sample-level metadata column we just gave it (`sample_id`), and `.var` carries the four protein-level columns we passed via `var_annotation`. From here on, every step adds to or filters this object.

In [ ]:
# Merge sample-level annotations
obs_ann = pd.read_csv(METADATA_PATH)

pr.ann.samples(
    adata,
    df=obs_ann,
    obs_on='sample_id',
    df_on='sample_id',
)

adata

In [ ]:
# Rename the group column for convenience
adata.obs = adata.obs.rename(columns={'group': 'tumor_class'})

adata.obs_names = adata.obs['sample_id'].astype(str)

## Quality control and preprocessing

First, we visualize the number of samples we have per tumor class.

In [ ]:
pr.pl.n_samples_per_category(adata, category_key=["tumor_class"])

> 📝 *Your turn*: Do we have enough samples for a two-group comparison? Are the tumor classes balanced?

<details>
<summary>Answer</summary>

The cohort spans several tumor classes; only the two with the largest *n* (here `EPN_MPE` and `EPN_SPINE`) are suitable for a parametric two-group test. A handful of classes are represented by 1–2 samples and we will drop them from the differential analysis — a t-test on n=2 has essentially no power and any "significant" result would be noise.

The two main groups are not perfectly balanced but are within a 2–3× ratio of each other, which is fine for a Welch t-test (it does not assume equal sample sizes or equal variance).

</details>

In [ ]:
adata = adata[adata.obs['tumor_class'].isin(['EPN_MPE', 'EPN_SPINE'])]

#### Contaminant removal

Sample preparation introduces non-biological proteins: keratins from skin/hair, trypsin used for digestion, BSA from buffers, etc. These co-purify with the sample and produce strong, sample-independent signals that can dominate downstream analysis if left in.

`pr.download.contaminants` fetches a curated FASTA of known contaminants and `pr.pp.remove_contaminants` strips any matching proteins from the AnnData object. Here we use the *Frankenfield 2022* list.

> 🔍 *Dig deeper*: read the [`pr.download.contaminants` docs](https://proteopy.readthedocs.io/en/latest/api/generated/proteopy.download.contaminants.html) to see the other contaminant sources `proteopy` ships with (e.g. GPM cRAP), and the original [Frankenfield et al. 2022 paper](https://pubs.acs.org/doi/10.1021/acs.jproteome.2c00145) for the rationale behind the updated list.

In [ ]:
contaminant_path = pr.download.contaminants(
    source='frankenfield2022',
    path='data/contaminants_frankenfield2022.fasta',
    force=True,
)

pr.pp.remove_contaminants(
    adata,
    contaminant_path=contaminant_path,
)

#### Abundance rank plot

The abundance rank plot sorts proteins per group from most to least intense and plots intensity against rank. It is a quick sanity check on the **dynamic range** captured by each group: a healthy proteomics experiment spans roughly 4–6 orders of magnitude, with a few very abundant proteins at the top and a long tail of low-abundance ones.

In [ ]:
pr.pl.abundance_rank(adata, color='tumor_class')

> 📝 *Your turn*: Compare the curves of the two tumor classes and interpret the plot.

#### Sample-wise protein coverage

Number of proteins quantified per sample is a direct readout of run quality. Samples that identify many fewer proteins than their peers usually had a failed digestion, low loading, or instrument issues and will introduce noise downstream. We drop any sample below `min_count=4000`.

In [ ]:
pr.pl.n_proteins_per_sample(
    adata,
    zero_to_na=True,
    order_by='tumor_class',
)

In [ ]:
pr.pp.filter_samples(
    adata,
    min_count=6000,
    zero_to_na=True,
)

#### Protein-wise completeness filtering

For each protein, we ask: *in what fraction of samples was it quantified?* Proteins measured in only a handful of samples cannot support a meaningful group comparison and contribute mostly imputation noise. We keep proteins seen in **≥ 80 %** of samples.

In [ ]:
pr.pl.completeness_per_var(adata, zero_to_na=True)

> 📝 *Your turn*: What fraction of proteins are quantified in *all* samples? What about the long tail of nearly-missing proteins? Justify the choice of `min_fraction=0.8` — what would change at 0.95?

In [ ]:
pr.pp.filter_var_completeness(adata, min_fraction=0.8, zero_to_na=True)

#### Within-group coefficient of variation

The coefficient of variation (CV = SD / mean) per protein within each group measures technical reproducibility. Most proteins should sit well below the 50 % reference line; a long tail above it indicates noisy quantification and is a red flag for the group's data quality.

In [ ]:
pr.pl.cv_by_group(
    adata,
    group_by="tumor_class",
    min_samples=3,
    hline=0.5,
)

> 📝 *Your turn*: Compare the CV distributions of the two groups. Is one noticeably noisier? Where does the bulk of each distribution sit relative to the 50 % reference line?

### Log transform

Most parametric statistical tests (t-test, ANOVA, linear models) assume **normally distributed** observations and **homoscedasticity** — variance that does not depend on the mean. Raw MS intensities violate both: they are approximately **log-normal**, and their noise scales with intensity (*heteroscedastic*: more abundant proteins are noisier on the linear scale).

A log2 transform fixes both problems at once: log-normal becomes roughly normal, and the multiplicative, intensity-dependent noise becomes additive and roughly constant across the dynamic range (*variance stabilization*). As a bonus, differences on the log2 scale read directly as fold-changes — a difference of 1 is a 2× change.

We keep the raw intensities in `adata.layers['raw']` so we can revisit them later if needed.

In [ ]:
# Convert zeros to NaN and log2 transform
adata.layers["raw"] = adata.X.copy()
adata.X[adata.X == 0] = np.nan
adata.X = np.log2(adata.X)

### Median normalization

Even after careful sample preparation, total signal varies between runs for technical reasons (loading amount, instrument response, column condition). **Median normalization** centers every sample's log-intensities on a common median, removing a multiplicative shift while leaving relative differences between proteins within a sample intact.

In [ ]:
# Intensity boxplot before normalization
pr.pl.intensity_box_per_sample(
    adata,
    order_by="tumor_class",
    zero_to_na=True,
)

In [ ]:
pr.pp.normalize_median(adata, log_space=True)

In [ ]:
# Intensity boxplot before normalization
pr.pl.intensity_box_per_sample(
    adata,
    order_by="tumor_class",
    zero_to_na=True,
)

> 📝 *Your turn*: Compare the boxplot above with the one before `pr.pp.normalize_median`. What changed? Did the within-sample spread change?

### Missing value imputation

Missing values are pervasive in DIA proteomics — a protein can be missing because it really is absent, or because its signal fell below the detection limit ("missing not at random", MNAR). Pretending they are zero distorts statistics; dropping them throws away signal.

The **downshift imputation** strategy, popularized by the [Perseus software](https://maxquant.net/perseus/) and the Cox lab, replaces missing values with draws from a Gaussian that is shifted *down* relative to each sample's measured distribution (here: 1.8 standard deviations below the mean, with a width of 0.3 SD). The intuition is that an unseen protein is most likely just below the detection limit — so we impute it there, with some random spread to avoid creating an artificial spike.

> 🔍 *Dig deeper*: read the [Perseus documentation on missing-value imputation](https://cox-labs.github.io/coxdocs/replacemissingfromgaussian.html) and consider when MNAR vs. MAR assumptions are appropriate. The histogram below colours imputed values so you can sanity-check the result.

In [ ]:
pr.pp.impute_downshift(
    adata,
    downshift=1.8,
    width=0.3,
    zero_to_na=True,
    random_state=123,
    verbose=True,
)

In [ ]:
# Combined histogram across all samples
pr.pl.intensity_hist(
    adata,
    color_imputed=True,
    density=False,
)

# Per-sample histograms (small multiples)
pr.pl.intensity_hist(
    adata,
    color_imputed=True,
    per_obs=True,
    ncols=5,
    legend_loc="upper right",
    density=False,
    figsize=(17, 12),
)

> 📝 *Your turn*: Where do the imputed values sit relative to the measured ones? Does the downshifted Gaussian fall in a plausible region for 'below detection limit', or does it overlap with the measured distribution? How does imputation differ between samples?

## Exploratory data analysis

Before any statistical model, we look at the data to understand the sources of variation: do replicates cluster together? Do tumor classes separate? Are there outlier samples? We use a sample-sample correlation heatmap, PCA, and UMAP as complementary views.

In [ ]:
pr.pl.sample_correlation_matrix(
    adata,
    margin_color="tumor_class",
)

> 📝 *Your turn*: Do samples of the same tumor class cluster together? Are there any obvious outliers correlating poorly with the rest?

<details>
<summary>Answer</summary>

The heatmap shows clear **block structure along the diagonal** that lines up with the tumor-class margin annotation — samples of the same class correlate more strongly with each other (typically Pearson *r* ≳ 0.9) than with samples of the other class. That is the first non-trivial sign that there is a real biological signal separating the two groups, before we have done any statistics.

Look for any single sample that shows uniformly *low* correlation with **everyone else, including members of its own class** — that is the outlier signature (single light row/column cutting through the heatmap). Here the correlations within class are high and homogeneous, so we have no obvious sample to drop.

If you did find an outlier, the next steps would be: (a) check its protein count and CV — was it already borderline at the QC stage? (b) cross-reference its acquisition / preparation metadata (not loaded in this trimmed annotation file, but worth re-querying from the LIMS) — is it from a different batch? (c) decide whether to drop it or to add the batch as a covariate.

</details>

In [ ]:
sc.tl.pca(adata)
with rc_context({"figure.figsize": (5, 3)}):
    sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

sc.pl.pca(
    adata,
    color="tumor_class",
    dimensions=(0, 1),
    ncols=2,
    size=100,
)

> 📝 *Your turn*: Does PC1 separate tumor classes? How much variance do the first few PCs explain? Is it reasonable to stop at PC2 or should we look further?

In [ ]:
sc.pp.neighbors(adata, n_neighbors=3)
sc.tl.umap(adata)

sc.pl.umap(
    adata,
    color="tumor_class",
    size=100,
)

> 📝 *Your turn*: Does UMAP separate the tumor classes more clearly than PCA? UMAP preserves local structure — what could small sub-clusters within a class represent?

<details>
<summary>Answer</summary>

UMAP usually shows a **crisper** separation than PCA on the same data, because it is a non-linear embedding that pulls similar samples tightly together and pushes dissimilar ones apart. Here the two tumor classes form two visibly distinct islands, often with no overlap between them.

The flip side of that crispness is that UMAP **distances are not interpretable as distances in the original feature space**: the white space between two clusters does not tell you "how different" they are, only that the algorithm chose to separate them. So do not read "the EPN_MPE cluster is twice as far from EPN_SPINE as in PCA" as a biological statement.

The little sub-clusters inside a class can represent several things, in rough order of plausibility:

1. **Biological substructure** — molecular subtypes within a tumor class, anatomical sub-location (e.g. cervical vs. lumbar spinal ependymoma), grade.
2. **Technical structure** — different sample-preparation batches or MS acquisition dates. With the trimmed annotation file used here (only `sample_id` + `tumor_class`) we cannot check this directly; if it matters in your own analysis, re-merge the relevant batch metadata into `adata.obs` and re-colour the UMAP.

Whichever it is, this is exactly the kind of structure that **the differential-abundance t-test will average over** if you do not account for it — so it is worth identifying before running statistics.

</details>

> 💻 *Code it*: UMAPs are a 2D representation of higher-dimensional data. The `n_neighbors` parameter controls the **local-vs-global trade-off**: small values pin the layout to very local structure, larger values pull it toward the global geometry.
>
> Rerun `sc.pp.neighbors` and `sc.tl.umap` with several different `n_neighbors` values (e.g. 2, 5, 10, 15, 30) and re-plot. How does the embedding change? Do the tumor classes separate more cleanly at any particular value?

In [ ]:
# Insert your code here

## Differential protein abundance

We now ask: *which proteins differ between two tumor classes?* Here we compare myxopapillary ependymoma of the spine (`EPN_MPE`) with non-myxopapillary spinal ependymoma (`EPN_SPINE`). We use a two-sample t-test in log-space and adjust p-values with Benjamini–Hochberg (BH) to control the false discovery rate. The volcano plot summarizes effect size (log2 fold change) against statistical significance.

Per-protein results are stored under `adata.varm[...]` keyed by the test design, and can be pulled out as a tidy data frame with `pr.get.differential_abundance_df`.

In [ ]:
pr.tl.differential_abundance(
    adata,
    method="ttest_two_sample",
    group_by="tumor_class",
    setup={"group1": "EPN_MPE", "group2": "EPN_SPINE"},
    multitest_correction="bh",
    alpha=0.01,
    space="log",
)

In [ ]:
pr.get.tests(adata)

In [ ]:
dpa = pr.get.differential_abundance_df(
    adata,
    keys='ttest_two_sample;tumor_class;EPN_MPE_vs_EPN_SPINE',
    sort_by='pval_adj',
)
dpa['gene_id'] = dpa['var_id'].map(adata.var['Genes'].to_dict())

In [ ]:
pr.pl.volcano(
    adata,
    varm_slot="ttest_two_sample;tumor_class;EPN_MPE_vs_EPN_SPINE",
    fc_thresh=0.5,
    pval_thresh=0.01,
    top_labels=5,
    alt_labels_key="Genes",
    xlabel="log2FC",
    ylabel="pval_adj",
)

> 📝 *Your turn*: How many proteins pass the dual significance + fold-change threshold? What are the top hits?

In [ ]:
# Insert your code here

### Set enrichment analysis using Decoupler

Lists of differentially abundant proteins are hard to interpret on their own. **Gene-set enrichment analysis (GSEA)** asks a question at a higher level: *is a predefined set of biologically related genes systematically up- or down-regulated in our comparison?* Instead of looking at single proteins, we look at coordinated shifts across whole pathways or signatures.

Three closely related methods you should know:

- **ORA** (Over-Representation Analysis): take the top-N significant genes, ask whether any pathway is enriched among them with a hypergeometric / Fisher test. Simple, but throws away the ranking and is sensitive to your significance cutoff.
- **GSEA** (Gene Set Enrichment Analysis, Subramanian et al. 2005): rank *all* genes by a statistic (e.g. the t-statistic), then test whether members of a pathway concentrate at the top or bottom of the ranked list. No arbitrary cutoff, uses the full signal.
- **GSVA** (Gene Set Variation Analysis): turns the gene × sample matrix into a pathway × sample matrix in an *unsupervised* way, useful for downstream clustering or per-sample pathway activity.

[`decoupler`](https://decoupler.readthedocs.io/) is a unified Python package that implements ORA, GSEA, GSVA and many other enrichment methods under a common API, and ships convenient access to curated gene-set databases via `dc.op` (the Omnipath bridge). We use it here to run GSEA against MSigDB.

Find the decoupler documentation [here](https://decoupler.readthedocs.io/en/latest/api/generated/decoupler.mt.gsea.html).

**MSigDB** — the [Molecular Signatures Database](https://www.gsea-msigdb.org/gsea/msigdb) — is the canonical collection of gene sets curated by the Broad Institute. It groups tens of thousands of gene sets into *collections* (hallmark pathways, canonical pathways such as Reactome/KEGG, GO terms, chemical and genetic perturbations, immunologic signatures, oncogenic signatures, cell-type markers, miRNA / TF target sets, etc.). The collection name appears in the `collection` column above.

In [ ]:
dc.op.show_resources()

In [ ]:
msig_db = dc.op.resource(
    name='MSigDB',
    organism='human',
    license='academic',
)

map = {
    'geneset': 'source',
    'genesymbol': 'target',
}
msig_db = msig_db.rename(columns=map)

msig_db = msig_db.drop_duplicates(subset=['source', 'target'])
msig_db

In [ ]:
tstats = dpa[['gene_id', 'tstat']].set_index('gene_id').drop_duplicates().T
tstats

In [ ]:
nes_scores, pval_adj = dc.mt.gsea(tstats, net=msig_db, tmin=3)

pval_adj = pval_adj.T.clip(2.22e-16, 1).T
mask = (pval_adj.T < 0.1).iloc[:, 0]
nes_scores = nes_scores.loc[:, mask]
pval_adj = pval_adj.loc[:, mask]

In [ ]:
dc.pl.barplot(
    data=nes_scores,
    name='tstat',
    top=50,
    figsize=(15,15),
)

> 📝 *Your turn*: Pick two or three biological themes from the top pathways. Do any make immediate sense for spinal vs. myxopapillary ependymoma?

In [ ]:
_, le = dc.pl.leading_edge(
    dpa.set_index('gene_id'),
    stat="tstat",
    net=msig_db,
    name="REACTOME_MITOCHONDRIAL_PROTEIN_DEGRADATION",
)
print("leading edge:", le[:5])

> 📝 *Your turn*: How do you interpret the plot? Pick one or two leading-edge genes for a pathway of interest and look them up. Do they have a known role in ependymoma or in the relevant biological process?

### 🧠 Task — what's wrong with this picture?

Before we summarize, take a moment and think:

> **What is the central methodological flaw in applying GSEA (as implemented above using MSigDB as reference gene set database) directly to a DIA-NN protein-level result?**

<details>
<summary>Hint:</summary>
Look closely at the `Genes` column of `adata.var` and at the `target` column of `msig_db`. What are the units in each? What happens when a single DIA-NN row corresponds to a **protein group** containing several proteins, each mapping to its own gene symbol? And what about a protein whose `Genes` field contains `GENEA;GENEB`?
</details>

Discuss with your neighbour:

1. How does our `tstats` table currently treat protein groups that map to multiple genes?
2. How does MSigDB treat the same gene appearing in different contexts?
3. What would a more rigorous mapping look like? (Keywords to look up: *peptide-centric* vs. *protein-centric* enrichment, *protein-group ambiguity*, *gene-symbol mapping*.)

<details>
<summary>Solution:</summary>
</details>

## Summary — what you have learned

Working through this notebook, you have:

- **Read a DIA-NN `report.parquet`** and filtered it on three complementary FDR statistics before pivoting to a sample × protein matrix.
- Met the **`proteopy` + `AnnData`** data model and seen how submodules (`pr.read` / `pp` / `pl` / `tl` / `get`) map cleanly onto analysis stages.
- Performed **quality control**: contaminant removal (Frankenfield 2022), low-coverage sample filtering, and completeness-based protein filtering.
- Applied a standard **preprocessing pipeline**: log2 transform → median normalization → downshift imputation (Perseus-style) for MNAR missing values.
- Explored the data with **sample correlation, PCA, and UMAP**.
- Computed **differential protein abundance** with a two-sample t-test and BH correction, and visualized it with a volcano plot.
- Run **gene-set enrichment (GSEA)** with `decoupler` against MSigDB, inspected the top pathways with bar- and dot-plots, and examined the **leading edge** to see which genes drive the enrichment.

You have also seen the conceptual limits: MS proteomics quantifies **protein groups**, not genes, and translating between the two — for gene-set enrichment in particular — is a step that requires care.

**Where to go next**

- Swap GSEA for ORA (`dc.mt.ora`) or GSVA (`dc.mt.gsva`) and compare the top hits.
- Replace MSigDB with a more focused resource (Reactome, PROGENy, DoRothEA) via `dc.op.resource(...)`.
- Read the [`proteopy`](https://proteopy.readthedocs.io/) and [`decoupler`](https://decoupler.readthedocs.io/) documentation for the full set of tools.

In [ ]:
!pip freeze